In [1]:
import os
os.chdir('../')

In [2]:
%pwd

'/Users/amith2831/Desktop/PROJECTS/MEDICAL CHATBOT'

In [3]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [4]:
# extract text from PDF Files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob='*.pdf',
        loader_cls=PyPDFLoader
    )

    documents = loader.load()

    return documents

In [5]:
extracted_data = load_pdf_files('data')

In [6]:
len(extracted_data)

637

In [7]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get('source')
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={'source': src}
            )
        )

    return minimal_docs

In [8]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [9]:
# split the data into chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [10]:
texts_chunk = text_split(minimal_docs)

In [11]:
print(f'Number of chunks: {len(texts_chunk)}')

Number of chunks: 5859


In [13]:
import torch
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

/var/folders/x7/brztb_5d3dj1h0rt_f56v83c0000gn/T/ipykernel_4093/3499970334.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [14]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [16]:
embedding_vector = embedding.embed_query('Hello there')
print(f'Dimension of vector: {len(embedding_vector)}')

Dimension of vector: 384


In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
import os
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [19]:
from pinecone import Pinecone
pincone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pincone_api_key)
pc

In [21]:
from pinecone import ServerlessSpec
index_name = 'medical-chatbot'

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )

index = pc.Index(index_name)

In [23]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [24]:
# # load existing index
# from langchain_pinecone import PineconeVectorStore
# docsearch = PineconeVectorStore.from_existing_index(
#     index_name=index_name,
#     embedding=embedding
# )

In [37]:
# adding more data into existing pinecone index
dswith = Document(
    page_content="amith kumar s is a data scientist based off bangalore," \
    " with 3+ years of experience, currently working on a job change",
    metadata={'source': 'youtube'}
)

docsearch.add_documents(documents=[dswith])

['35fe7a56-bb3d-4156-b0b3-7c3c0fe32f9c']

In [38]:
retriever = docsearch.as_retriever(
    search_type='similarity',
    search_kwargs={'k':3}
)

retriever_docs = retriever.invoke('What is acne?')

In [39]:
retriever_docs

[Document(id='e2e36257-8f52-4df2-a03b-2a1cb00d9a7a', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='d76b23d5-5027-4983-b087-df896b12786d', metadata={'source': 'data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='87a6ef65-bda8-4046-9a85-9779ce3043ac', metadata={'source': 'data/Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with 

In [40]:
from langchain_openai import ChatOpenAI
chatmodel = ChatOpenAI(
    model='gpt-4o'
)

In [41]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', system_prompt),
        ('human', '{input}')
    ]
)

In [42]:
question_answer_chain = create_stuff_documents_chain(chatmodel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [43]:
response = rag_chain.invoke({'input': 'What is acromegaly and gigantism?'})
print(response['answer'])

Acromegaly is a disorder caused by the abnormal release of growth hormone from the pituitary gland, leading to increased growth in bone and soft tissue, along with other disturbances in the body. It occurs after bone growth has stopped and typically presents in middle-aged individuals. Gigantism, on the other hand, occurs when excess growth hormone is released before bone growth has stopped, resulting in unusual height.


In [44]:
response = rag_chain.invoke({'input': 'What is acne?'})
print(response['answer'])

Acne is a common skin disease characterized by pimples appearing on the face, chest, and back. It occurs when the skin's pores become clogged with oil, dead skin cells, and bacteria. The medical term for common acne is acne vulgaris.


In [45]:
response = rag_chain.invoke({'input': 'What is androgenic alopecia?'})
print(response['answer'])

Androgenic alopecia, also known as male pattern baldness, is a common form of hair loss in adult males. It is characterized by hair loss over the top and front of the scalp. This condition is considered a normal pattern of hair loss and can be identified by the distribution of the hair loss.


In [47]:
response = rag_chain.invoke({'input': 'who is amith?'})
print(response['answer'])

Amith Kumar S is a data scientist based in Bangalore with over 3 years of experience and is currently seeking a job change.
